# Merge Rate to Length of Time on Project - Visualization

This visualization notebook implements the prototype for 8Knot and CHAOSS:
* **8Knot Feature Request**: [8Knot #710 - Merge rate to length of time on project](https://github.com/oss-aspen/8Knot/issues/710)
* **CHAOSS Working Group Issue**: [wg-data-science #201](https://github.com/chaoss/wg-data-science/issues/201)
* **CHAOSS Metric Reference**: [Change Request Acceptance Ratio](https://chaoss.community/kb/metric-change-request-acceptance-ratio/)

### Metric Overview
Understanding how contributor tenure (the length of time a person has been active in a repository) influences their pull request acceptance rate is essential for community sustainability and contributor onboarding.

This notebook analyzes:
1. **Tenure Cohorts**: Categorizing pull requests by the author's tenure at the time of submission (First Contribution, < 30 Days, 1 - 6 Months, 6 - 12 Months, 1+ Years).
2. **PR Volume by Tenure**: Total submitted versus merged pull requests across each tenure tier.
3. **Merge Acceptance Rate (%)**: The proportion of PRs merged across tenure tiers (`(Merged PRs / Submitted PRs) * 100`) against the repository baseline.
4. **Merge Velocity & Review Friction**: Time-to-merge distribution (in days) across tenure cohorts to evaluate if newer contributors face longer review cycles.

### Importing Libraries

In [ ]:
import sys
import os
import json
from pathlib import Path
import datetime as dt
import pandas as pd
import numpy as np
import sqlalchemy as salc
import plotly.graph_objects as go
import plotly.express as px

## Installing Dependencies - Package Management

* This particular implementation uses requirements.txt to install dependencies if needed

In [ ]:
# Package management - Checking requirements.txt
_repo_root = Path.cwd().parent if Path.cwd().name == "8knot" else Path.cwd()
_req_file = _repo_root / "requirements.txt"
if _req_file.exists():
    print(f"requirements.txt located at: {_req_file}")

### Loading Credentials - .env; .json...

In [ ]:
# augur_creds.json lives in the same folder as this notebook (8knot/).
# If Jupyter's working directory is collectoss-analysis/ instead, fall back to 8knot/augur_creds.json.
_cred_path = Path("augur_creds.json")
if not _cred_path.is_file():
    _cred_path = Path("8knot") / "augur_creds.json"
if not _cred_path.is_file():
    raise FileNotFoundError(
        "augur_creds.json should sit next to merge_rate_by_tenure.ipynb. "
        f"Not found from cwd: {Path.cwd().resolve()}"
    )
with open(_cred_path) as config_file:
    config = json.load(config_file)

### DB connection

In [ ]:
database_connection_string = "postgresql+psycopg2://{}:{}@{}:{}/{}".format(
    config["user"], config["password"], config["host"], config["port"], config["database"]
)
dbschema = "augur_data"
engine = salc.create_engine(
    database_connection_string,
    connect_args={"options": "-csearch_path={}".format(dbschema)},
)

### Queries

Get repo_ids and repo_names from augur data

In [ ]:
repo_urls = ["https://github.com/chaoss/augur"]

url_query = str(repo_urls)[1:-1]
repo_query = salc.sql.text(
    f"""
    SET SCHEMA 'augur_data';
    SELECT DISTINCT r.repo_id, r.repo_name
    FROM repo r
    JOIN repo_groups rg ON r.repo_group_id = rg.repo_group_id
    WHERE r.repo_git IN ({url_query})
    """
)
with engine.connect() as conn:
    t = conn.execute(repo_query)
    results = t.all()
repo_ids = [row[0] for row in results]
repo_names = [row[1] for row in results]
print(repo_ids)
print(repo_names)

[36113]
['augur']


### Query for PRs and Contributor First Contribution Date

Queries all pull requests alongside the contributor's first recorded PR timestamp on the repository to compute contributor tenure at the time of each PR creation.

In [ ]:
repo_statement = str(repo_ids)[1:-1]

query = salc.sql.text(f"""
    WITH first_contrib AS (
        SELECT
            pr.pr_augur_contributor_id AS cntrb_id,
            pr.repo_id,
            MIN(pr.pr_created_at) AS first_pr_at
        FROM pull_requests pr
        WHERE pr.repo_id IN ({repo_statement})
        GROUP BY pr.pr_augur_contributor_id, pr.repo_id
    )
    SELECT
        r.repo_id,
        r.repo_name,
        pr.pull_request_id,
        pr.pr_src_number,
        left(pr.pr_augur_contributor_id::text, 15) AS cntrb_id,
        pr.pr_created_at AS created_at,
        pr.pr_closed_at AS closed_at,
        pr.pr_merged_at AS merged_at,
        fc.first_pr_at AS first_contribution_at
    FROM repo r
    JOIN pull_requests pr ON r.repo_id = pr.repo_id
    JOIN first_contrib fc ON pr.pr_augur_contributor_id = fc.cntrb_id AND pr.repo_id = fc.repo_id
    WHERE r.repo_id IN ({repo_statement})
        AND pr.pr_created_at < now()
        AND (pr.pr_closed_at < now() OR pr.pr_closed_at IS NULL)
        AND (pr.pr_merged_at < now() OR pr.pr_merged_at IS NULL)
    ORDER BY pr.pr_created_at;
"""
)
df = pd.read_sql(query, con=engine)
df = df.reset_index()
df.drop("index", axis=1, inplace=True)

df["created_at"] = pd.to_datetime(df["created_at"], utc=True)
df["closed_at"] = pd.to_datetime(df["closed_at"], utc=True)
df["merged_at"] = pd.to_datetime(df["merged_at"], utc=True)
df["first_contribution_at"] = pd.to_datetime(df["first_contribution_at"], utc=True)
df.head()

,repo_id,repo_name,pull_request_id,pr_src_number,cntrb_id,created_at,closed_at,merged_at,first_contribution_at
0,36113,augur,1104462,62,01000a11-4000-0,2022-01-11 00:00:00+00:00,2022-01-20 02:10:43.084387+00:00,2022-01-20 02:10:43.084387+00:00,2022-01-11 00:00:00+00:00
1,36113,augur,1104636,236,01000a43-4000-0,2022-01-18 00:00:00+00:00,NaT,NaT,2022-01-18 00:00:00+00:00
2,36113,augur,1104719,319,01000a59-4000-0,2022-01-23 00:00:00+00:00,2022-02-04 12:58:07.102597+00:00,2022-02-04 12:58:07.102597+00:00,2022-01-23 00:00:00+00:00
3,36113,augur,1104569,169,01000a30-4000-0,2022-01-24 00:00:00+00:00,2022-02-03 09:52:42.391448+00:00,2022-02-03 09:52:42.391448+00:00,2022-01-24 00:00:00+00:00
4,36113,augur,1104431,31,01000a07-4000-0,2022-01-30 00:00:00+00:00,2022-02-15 17:35:14.192006+00:00,2022-02-15 17:35:14.192006+00:00,2022-01-30 00:00:00+00:00


### Data Processing: Calculating Contributor Tenure & Cohorts

Computes contributor tenure (in days) at the exact time of PR submission and segments PRs into standard lifecycle cohorts:
* **First Contribution (Day 0)**: The author's very first pull request on the repository.
* **< 30 Days**: Early-stage newcomers within their first month of activity.
* **1 - 6 Months**: Developing contributors gaining familiarity with the codebase.
* **6 - 12 Months**: Established contributors with sustained project participation.
* **1+ Years**: Veteran contributors with over a year of repository history.

In [ ]:
# Calculate tenure in days at PR creation time
df["tenure_days"] = (df["created_at"] - df["first_contribution_at"]).dt.total_seconds() / 86400.0
df["is_merged"] = df["merged_at"].notna()
df["merge_duration_days"] = (df["merged_at"] - df["created_at"]).dt.total_seconds() / 86400.0

# Define tenure cohorts
def assign_tenure_cohort(days):
    if days <= 0.0001:
        return "1. First PR (Day 0)"
    elif days < 30:
        return "2. < 30 Days"
    elif days < 180:
        return "3. 1 - 6 Months"
    elif days < 365:
        return "4. 6 - 12 Months"
    else:
        return "5. 1+ Years"

df["tenure_cohort"] = df["tenure_days"].apply(assign_tenure_cohort)

print("Pull request distribution across contributor tenure cohorts:")
print(df["tenure_cohort"].value_counts().sort_index())

Pull request distribution across contributor tenure cohorts:
tenure_cohort
1. First PR (Day 0)     70
2. < 30 Days            23
3. 1 - 6 Months         83
4. 6 - 12 Months        84
5. 1+ Years            132


### Metric Aggregation: Merge Rate by Tenure Tier

Aggregates submitted volume, merged volume, merge percentage (`%`), and median turnaround time for each tenure cohort.

In [ ]:
tenure_summary = df.groupby("tenure_cohort").agg(
    prs_submitted=("pull_request_id", "count"),
    prs_merged=("is_merged", "sum"),
    median_time_to_merge=("merge_duration_days", lambda x: np.round(np.nanmedian(x), 2) if np.any(np.isfinite(x)) else 0.0)
).reset_index()

tenure_summary["merge_rate_pct"] = np.round(
    (tenure_summary["prs_merged"] / tenure_summary["prs_submitted"]) * 100, 1
)

tenure_summary = tenure_summary.sort_values(by="tenure_cohort").reset_index(drop=True)
print("=== Summary of PR Activity by Contributor Tenure ===")
tenure_summary

=== Summary of PR Activity by Contributor Tenure ===


,tenure_cohort,prs_submitted,prs_merged,median_time_to_merge,merge_rate_pct
0,1. First PR (Day 0),70,38,6.15,54.3
1,2. < 30 Days,23,18,6.21,78.3
2,3. 1 - 6 Months,83,55,6.46,66.3
3,4. 6 - 12 Months,84,75,2.91,89.3
4,5. 1+ Years,132,120,3.42,90.9


### Plots
## 1. PRs Submitted vs. Merged Across Tenure Tiers

Grouped bar chart showing pull request submission volume and merge volume across contributor tenure tiers.

In [ ]:
fig_volume = go.Figure()

fig_volume.add_trace(
    go.Bar(
        name="PRs Submitted",
        x=tenure_summary["tenure_cohort"],
        y=tenure_summary["prs_submitted"],
        marker_color="#2b5c8f",
        text=tenure_summary["prs_submitted"],
        textposition="auto",
        hovertemplate="<b>%{x}</b><br>Submitted PRs: %{y}<extra></extra>"
    )
)

fig_volume.add_trace(
    go.Bar(
        name="PRs Merged",
        x=tenure_summary["tenure_cohort"],
        y=tenure_summary["prs_merged"],
        marker_color="#2ca02c",
        text=tenure_summary["prs_merged"],
        textposition="auto",
        hovertemplate="<b>%{x}</b><br>Merged PRs: %{y}<extra></extra>"
    )
)

fig_volume.update_layout(
    title="Pull Request Volume (Submitted vs Merged) by Contributor Tenure",
    xaxis_title="Contributor Tenure Cohort",
    yaxis_title="Number of Pull Requests",
    barmode="group",
    font=dict(size=14),
    template="plotly_white",
    legend=dict(x=0.8, y=1.0)
)

fig_volume.show()

Figure({ 'data': [...] })

## 2. PR Merge Acceptance Rate (%) by Contributor Tenure

Bar and line comparison illustrating how PR acceptance probability evolves from first-time contributors to long-term veterans.

In [ ]:
overall_avg_rate = np.round((df["is_merged"].sum() / len(df)) * 100, 1)

fig_rate = go.Figure()

fig_rate.add_trace(
    go.Bar(
        name="Merge Rate (%)",
        x=tenure_summary["tenure_cohort"],
        y=tenure_summary["merge_rate_pct"],
        marker_color="#1f77b4",
        text=[f"{v:.1f}%" for v in tenure_summary["merge_rate_pct"]],
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>Merge Rate: %{y:.1f}%<extra></extra>"
    )
)

fig_rate.add_trace(
    go.Scatter(
        name="Trend",
        x=tenure_summary["tenure_cohort"],
        y=tenure_summary["merge_rate_pct"],
        mode="lines+markers",
        line=dict(color="#ff7f0e", width=3),
        marker=dict(size=8),
        hovertemplate="<b>Trend</b>: %{y:.1f}%<extra></extra>"
    )
)

fig_rate.add_hline(
    y=overall_avg_rate,
    line_dash="dash",
    line_color="#d62728",
    annotation_text=f"Repo Avg: {overall_avg_rate}%",
    annotation_position="top left"
)

fig_rate.update_layout(
    title="PR Merge Acceptance Rate (%) by Contributor Tenure Cohort",
    xaxis_title="Contributor Tenure Cohort",
    yaxis_title="Merge Rate (%)",
    font=dict(size=14),
    template="plotly_white",
    yaxis=dict(range=[0, 105]),
    legend=dict(x=0.02, y=0.98)
)

fig_rate.show()

Figure({ 'data': [...] })

## 3. Merge Rate vs. Tenure Duration (Continuous Binned Curve)

Evaluates the continuous relationship between contributor tenure (in 30-day windows up to 2 years) and the probability of merge.

In [ ]:
# 30-day window binned analysis
df_window = df.copy()
df_window["tenure_bin_days"] = (df_window["tenure_days"] // 30) * 30
df_window = df_window[df_window["tenure_bin_days"] <= 720]

window_agg = df_window.groupby("tenure_bin_days").agg(
    total=("pull_request_id", "count"),
    merged=("is_merged", "sum")
).reset_index()

window_agg["merge_rate_pct"] = np.round((window_agg["merged"] / window_agg["total"]) * 100, 1)

fig_continuous = px.scatter(
    window_agg,
    x="tenure_bin_days",
    y="merge_rate_pct",
    size="total",
    trendline="ols",
    title="PR Merge Rate vs. Contributor Tenure (30-Day Windows)",
    labels={"tenure_bin_days": "Contributor Tenure (Days)", "merge_rate_pct": "Merge Rate (%)", "total": "Total PRs"},
    template="plotly_white"
)

fig_continuous.update_layout(
    xaxis_title="Contributor Tenure at PR Submission (Days)",
    yaxis_title="Merge Acceptance Rate (%)",
    font=dict(size=14)
)

fig_continuous.show()

Figure({ 'data': [...] })

## 4. Time-to-Merge Distribution Across Tenure Cohorts

Box plot evaluating whether pull requests from longer-tenured contributors experience shorter review durations or expedited merging compared to newcomers.

In [ ]:
merged_df = df[df["is_merged"]].sort_values("tenure_cohort")

fig_box = px.box(
    merged_df,
    x="tenure_cohort",
    y="merge_duration_days",
    color="tenure_cohort",
    points="outliers",
    title="Time-to-Merge Distribution (Days) Across Contributor Tenure Cohorts",
    labels={"tenure_cohort": "Tenure Cohort", "merge_duration_days": "Duration to Merge (Days)"},
    template="plotly_white"
)

fig_box.update_layout(
    xaxis_title="Contributor Tenure Cohort",
    yaxis_title="Days to Merge",
    font=dict(size=14),
    showlegend=False
)

fig_box.show()

Figure({ 'data': [...] })

### Summary & Key Observations
* **Newcomer Friction**: First-time contributors often experience lower acceptance ratios, pointing to onboarding or styling hurdles that community maintainers can target with better documentation and CI guidance.
* **Core Contributor Velocity**: Long-tenured contributors show both higher merge probability and shorter review cycles due to established trust, architectural familiarity, and reviewer familiarity.
* **8Knot Dashboard Integration**: This visualization prototype maps directly into the 8Knot Contributions view (`8Knot/pages/contributions/`) to provide maintainers an ongoing health check on contributor onboarding efficiency.